In [0]:
from datetime import datetime
from pyspark.sql import SparkSession ,Row

CONTROL_TABLE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_control/ingestion_log"


In [0]:

def _ensure_control_table_exists(spark: SparkSession) -> None:
    
    if not spark._jsparkSession.catalog().tableExists("ingestion_log"):
        try:
            spark.read.format("delta").load(CONTROL_TABLE_PATH)
        except Exception:
            empty_df = spark.createDataFrame([], schema="""
                batch_id STRING,
                source_name STRING,
                partition_key STRING,
                status STRING,
                rows_written LONG,
                started_at TIMESTAMP,
                completed_at TIMESTAMP,
                error_message STRING
            """)
            empty_df.write.format("delta").mode("overwrite").save(CONTROL_TABLE_PATH)


In [0]:
def already_ingested(spark: SparkSession, source_name: str, partition_key: str) -> bool:

    _ensure_control_table_exists(spark)
    df = spark.read.format("delta").load(CONTROL_TABLE_PATH)
    match_count = (
        df.filter(
            (df.source_name == source_name)
            & (df.partition_key == partition_key)
            & (df.status == "SUCCESS")
        ).count()
    )
    return match_count > 0

In [0]:
def log_ingestion_event(
    spark: SparkSession,
    batch_id: str,
    source_name: str,
    partition_key: str,
    status: str,
    rows_written: int = 0,
    started_at: datetime = None,
    error_message: str = None,
) -> None:

    _ensure_control_table_exists(spark)
    row = Row(
        batch_id=batch_id,
        source_name=source_name,
        partition_key=partition_key,
        status=status,
        rows_written=rows_written,
        started_at=started_at or datetime.utcnow(),
        completed_at=datetime.utcnow(),
        error_message=error_message,
    )
    spark.createDataFrame([row]).write.format("delta").mode("append").save(CONTROL_TABLE_PATH)